# LAB | Abstractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [ ]:
# We pin exact versions of every library (including torch/torchvision, installed
# from PyTorch's official CUDA 12.1 wheels) so this notebook behaves the same way
# every time it runs, on any machine with a GPU.
!pip install -qU torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -qU datasets==2.21.0 huggingface_hub==0.25.2 pinecone-client==3.1.0 sentence-transformers==3.0.1 transformers==4.44.2

In [3]:
import sys
import torch
import transformers
import tokenizers
import datasets
import sentence_transformers
from pinecone import Pinecone, ServerlessSpec

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Datasets:", datasets.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("GPU available:", torch.cuda.is_available())

print("\nDependency imports passed.")

Python: 3.13.18
PyTorch: 2.11.0+cpu
Transformers: 4.57.6
Tokenizers: 0.22.2
Datasets: 3.6.0
Sentence Transformers: 3.4.1
GPU available: False

Dependency imports passed.


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [ ]:
from datasets import load_dataset

# We use the "wiki_snippets" dataset, which contains millions of short passages
# extracted from Wikipedia. Each passage comes with useful metadata: the article
# it was taken from (article_title), the section within that article (section_title),
# and the passage's text itself (passage_text). We use the "wiki40b_en_100_0"
# configuration, which contains English passages of ~100 words each.
wiki_data = load_dataset(
    'community-datasets/wiki_snippets',
    name='wiki40b_en_100_0',
    split='train',
    streaming=True
).shuffle(seed=960)

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [2]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'_id': '{"datasets_id": 1251236, "wiki_id": "Q2049363", "sp": 16, "sc": 56, "ep": 16, "ec": 109}',
 'datasets_id': 1251236,
 'wiki_id': 'Q2049363',
 'start_paragraph': 16,
 'start_character': 56,
 'end_paragraph': 16,
 'end_character': 109,
 'article_title': 'Panemotichus',
 'section_title': 'Site',
 'passage_text': 'Archaeologists have revealed Iron Age remains there.'}

In [3]:
# filter only documents with History as section_title - Replace None with your code
history = wiki_data.filter(
    lambda document: document["section_title"] == "History"
)

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [ ]:
from itertools import islice
from tqdm.auto import tqdm

# Start small to check that everything works.
total_doc_count = 1000

docs = []

for d in tqdm(
    islice(history, total_doc_count),
    total=total_doc_count,
    desc="Collecting history passages"
):
    docs.append({
        "article_title": d["article_title"],
        "section_title": d["section_title"],
        "passage_text": d["passage_text"]
    })

print(f"Collected {len(docs)} passages.")

In [6]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
print(f"Number of rows: {len(df)}")
df.head()

Number of rows: 1000


,article_title,section_title,passage_text
0,Paffendorf,History,Paffendorf History Ramon Zenker was involved i...
1,Peranakan,History,the Chinese was widespread.\nIt cannot be deni...
2,Pennsylvania Route 244,History,244 to its original alignment.
3,Patio 29,History,"kidnapped during the 1973 coup.\nIn 1981, Sant..."
4,Pecan Street Festival,History,Pecan Street Festival History This downtown Au...


In [9]:
%pip install -q "pinecone-client==3.1.0"

# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [10]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key,
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager
# (the key icon in the left sidebar).
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [11]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

print(f"Cloud: {cloud}")
print(f"Region: {region}")

Cloud: aws
Region: us-east-1


Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [12]:
index_name = "abstractive-question-answering" # choose a meaningful name for your index

In [13]:
import time

# check if index already exists (it shouldn't if this is first time)
#initialize the index, and insure the stats are all zeros

existing_indexes = pc.list_indexes().names()

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=spec
    )

    print("Creating index. Waiting until it is ready...")

    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

    print("Index is ready.")
else:
    print(f"Connecting to existing index: {index_name}")

index = pc.Index(index_name)

print(index.describe_index_stats())

Creating index. Waiting until it is ready...
Index is ready.
{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from huggingface model hub
#load the retriever model from HuggingFace. Use the flax-sentence-embeddings/all_datasets_v3_mpnet-base model
retriever = SentenceTransformer(
    "flax-sentence-embeddings/all_datasets_v3_mpnet-base",
    device=device
)
retriever

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [15]:
batch_size = 64

for i in tqdm(
    range(0, len(df), batch_size),
    desc="Creating and uploading embeddings"
):
    # Select the current batch.
    batch = df.iloc[i:i + batch_size]

    # Get the passage text from each row.
    passages = batch["passage_text"].tolist()

    # Convert passages into embeddings.
    embeddings = retriever.encode(
        passages,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # Create the vectors and metadata Pinecone needs.
    vectors = []

    for row_index, (_, row) in enumerate(batch.iterrows()):
        vectors.append({
            "id": str(i + row_index),
            "values": embeddings[row_index].tolist(),
            "metadata": {
                "article_title": row["article_title"],
                "section_title": row["section_title"],
                "passage_text": row["passage_text"]
            }
        })

    # Upload the batch.
    index.upsert(vectors=vectors)

print("Finished uploading vectors.")

# Pinecone's vector count may take a little time to update.
print(index.describe_index_stats())

Creating and uploading embeddings:   0%|          | 0/16 [00:00<?, ?it/s]

Finished uploading vectors.
{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 1000}},
 'total_vector_count': 1000}


# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [17]:
def query_pinecone(query, top_k):
    # Convert the question into an embedding.
    query_embedding = retriever.encode(query).tolist()

    # Search for the most similar passages.
    result = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return result

In [18]:
def format_query(query, context):
    # Add the <P> tag before each retrieved passage.
    passages = [
        f"<P> {match['metadata']['passage_text']}"
        for match in context
    ]

    # Join all passages into one string.
    combined_context = " ".join(passages)

    # Format the input expected by the generator.
    formatted_query = f"question: {query} context: {combined_context}"

    return formatted_query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [19]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

{'matches': [{'id': '113',
              'metadata': {'article_title': 'Pennsylvania Public Utility '
                                            'Commission',
                           'passage_text': 'in the way that the electricity '
                                           'was priced and regulated.',
                           'section_title': 'History'},
              'score': 0.609204292,
              'values': []}],
 'namespace': '',
 'usage': {'egressBytes': 184.0, 'read_units': 1}}

In [20]:
from pprint import pprint

In [21]:
# format the query in the form generator expects the input
query = format_query(query, result["matches"])
pprint(query)

('question: when was the first electric power system built? context: <P> in '
 'the way that the electricity was priced and regulated.')


The output looks great. Now let's write a function to generate answers.

In [23]:
def generate_answer(query):
    # Convert the formatted text into tokens.
    inputs = tokenizer(
        query,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Generate an answer without calculating gradients.
    with torch.no_grad():
        ids = generator.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            num_beams=2,
            min_length=20,
            max_length=40
        )

    # Convert the generated tokens back into text.
    answer = tokenizer.decode(
        ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    print(answer)

In [24]:
generate_answer(query)

The first electric power system was built in the early 19th century. The first electric power system was built in the early 20th century. The first electric power system was built in the 1920s


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [25]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

The first wireless message was sent in 1901 by Guglielmo Marconi, who was the inventor of the first wireless telegraph. The first wireless telegraph was built in 1901, and


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [26]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

the mass manufacturing techniques of the auto industry, Pasin began making metal wagons out of stamped steel in 1927. At around that time, the red wagons sold for slightly less than $3, or about $40 in 2016 dollars.
In 1930, the company was renamed Radio Steel & Manufacturing. The renamed company produced steel-bodied wagons and used assembly line manufacturing techniques. The new Radio Flyer wagons were named as a tribute to two famous men of the day: Marconi and Lindbergh. Italian inventor and engineer Guglielmo Marconi developed, demonstrated, and marketed the first successful long-distance wireless telegraph and in 1901 broadcast the
---
facilities-sharing agreement with competitor Smart Communications to be able to provide wireless services using Smart's GSM network. By April 2000, Piltel launched its GSM brand, Talk 'N Text.
Since then, most of Piltel's employees have been absorbed by Smart Communications. Likewise, the management of its wireless business have also been outsource

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [27]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

COVID-19 is a strain of the coronavirus. The coronavirus is the virus that causes coronavirus. The coronavirus is the virus that causes coronavirus.


In [28]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

The rest of the laboratories then became operational one by one. The official opening took place in 1999.
Since then, the laboratories in the facility have been instrumental in responding to a number of significant infectious disease outbreaks: The 2003 SARS outbreak when NML led the laboratory response; the 2003 BSE case when NCFAD provided the diagnosis of the initial case and then undertook testing as part of a vast investigation; the 2004 avian influenza outbreak in BC for which NCFAD led the laboratory investigation; and the 2009 pandemic H1N1 influenza where Mexico sought help from NML in identifying the unknown
---
population was much reduced by the black plague and subsequently by tuberculosis and cholera. The population grew rapidly from about 1800 and was supported by the economy of wine production from the 1820s until the phylloxera infection of the vines in the 1890s. Phylloxera was brought through imported vines from the United States, where it is endemic. All French vines

Let’s finish with a final few questions.

In [29]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

The War of Currents was a war between two competing currents. The two currents were the two-phase electric and magnetic fields. The electric field was the dominant force in the world, and the


In [30]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

The first man to walk on the moon was Neil Armstrong, who walked on the moon in 1969.


In [31]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

The Space Shuttle was the most expensive project in the history of the US government. It cost about $1.5 billion to build.


<br>

### 🚀 Bonus Challenges

Try one or more of these to practice:

- Ask your own questions and compare answers using different `top_k` values.
- Ask a question the dataset likely can't answer and observe how the model responds.
- Change the `section_title` filter to a topic other than "History" and rebuild a small index.
- Print the retrieved passages next to each generated answer to see how retrieval quality affects the result.
- Swap the retriever or generator for a different model from Hugging Face and compare outputs.

In [32]:
# Bonus 1: Compare answers using different top_k values.

question = "What caused the French Revolution?"

for top_k in [1, 3, 5]:
    print(f"\n{'=' * 60}")
    print(f"Question: {question}")
    print(f"top_k = {top_k}")

    # Retrieve passages from the existing Pinecone index.
    results = query_pinecone(question, top_k=top_k)
    matches = results["matches"]

    if not matches:
        print("No passages were retrieved.")
        continue

    # Show the evidence before generating the answer.
    print("\nRetrieved passages:")

    for number, match in enumerate(matches, start=1):
        metadata = match["metadata"]

        print(f"\nPassage {number}")
        print(f"Article: {metadata['article_title']}")
        print(f"Similarity score: {match['score']:.3f}")
        print(metadata["passage_text"])

    # Format the question and generate an answer.
    formatted_question = format_query(question, matches)

    print("\nGenerated answer:")
    generate_answer(formatted_question)


Question: What caused the French Revolution?
top_k = 1

Retrieved passages:

Passage 1
Article: Roquebrun
Similarity score: 0.376
population was much reduced by the black plague and subsequently by tuberculosis and cholera. The population grew rapidly from about 1800 and was supported by the economy of wine production from the 1820s until the phylloxera infection of the vines in the 1890s. Phylloxera was brought through imported vines from the United States, where it is endemic. All French vines are now grafted onto Native American grape rootstock. Rail transport was the means to distribute the wine internally and to the ports. Spanish workers began to settle in Roquebrun from about 1850 and refugees from the civil war settled in

Generated answer:
The French Revolution was not the result of a single event, but rather a series of events. The French Revolution was the culmination of a series of events that had been going on for centuries. The

Question: What caused the French Revolutio

In [33]:
# Find a passage with enough detail to create an answerable question.
examples = df[
    df["passage_text"].str.len() > 400
].head(3)

for _, row in examples.iterrows():
    print("Article:", row["article_title"])
    print(row["passage_text"])
    print("\n---\n")

Article: Paffendorf
Paffendorf History Ramon Zenker was involved in a number of dance music projects in the late 80's, including Interactive ("Who Is Elvis"). Engel's and Zenker's Samba dance project Bellini, centered around three female singers, began in 1997 with the release of "Samba de Janeiro" in various versions. It became an international hit (#2 in Germany) and was followed by an album and more singles.
The trance project Paffendorf debuted with "Smile" (#22 in Germany) in 1998. "Ruf Mich An" / "Call Me Now" and "Terminator" achieved Top 50 status. In the year 2000 they continued in that vein and made their

---

Article: Peranakan
the Chinese was widespread.
It cannot be denied, however, that the existence of slavery in this quarter, in former years, was of immense advantage in procuring a female population for Pinang. From Assaban alone, there used to be sometimes 300 slaves, principally females, exported to Malacca and Pinang in a year. The women get comfortably settled as t

In [35]:
# Bonus 1 continued: Compare top_k with an answerable question.

question = "What song did Paffendorf debut with in 1998?"

for top_k in [1, 3, 5]:
    print(f"\n{'=' * 60}")
    print(f"Question: {question}")
    print(f"top_k = {top_k}")

    results = query_pinecone(question, top_k=top_k)
    matches = results["matches"]

    if not matches:
        print("No passages were retrieved.")
        continue

    print("\nRetrieved passages:")

    for number, match in enumerate(matches, start=1):
        metadata = match["metadata"]

        print(f"\nPassage {number}")
        print(f"Article: {metadata['article_title']}")
        print(f"Similarity score: {match['score']:.3f}")
        print(metadata["passage_text"])

    formatted_question = format_query(question, matches)

    print("\nGenerated answer:")
    generate_answer(formatted_question)


Question: What song did Paffendorf debut with in 1998?
top_k = 1

Retrieved passages:

Passage 1
Article: Paffendorf
Similarity score: 0.614
Paffendorf History Ramon Zenker was involved in a number of dance music projects in the late 80's, including Interactive ("Who Is Elvis"). Engel's and Zenker's Samba dance project Bellini, centered around three female singers, began in 1997 with the release of "Samba de Janeiro" in various versions. It became an international hit (#2 in Germany) and was followed by an album and more singles.
The trance project Paffendorf debuted with "Smile" (#22 in Germany) in 1998. "Ruf Mich An" / "Call Me Now" and "Terminator" achieved Top 50 status. In the year 2000 they continued in that vein and made their

Generated answer:
I'm not sure if this is what you're looking for, but here's a sample of the song:

Question: What song did Paffendorf debut with in 1998?
top_k = 3

Retrieved passages:

Passage 1
Article: Paffendorf
Similarity score: 0.614
Paffendorf H

Observation: The correct passage was retrieved first for all three top_k values. It explicitly states that Paffendorf debuted with “Smile” in 1998. However, the generator did not answer the question and instead produced irrelevant responses about a sample or music video. Increasing top_k did not improve accuracy. This demonstrates that successful retrieval does not guarantee a correct generated answer.

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
# To delete the Pinecone index,
# uncomment the line below + run this code cell

# pc.delete_index(index_name)